In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os, re, ast, json, random, argparse, sys, hashlib, math
from typing import List, Dict, Any, Tuple, Union, Optional
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer, AutoModel, AutoModelForSequenceClassification,
    Trainer, TrainingArguments, EarlyStoppingCallback, get_linear_schedule_with_warmup
)
from sklearn.metrics import f1_score, jaccard_score, precision_score, recall_score

# ============================ CONFIG (defaults) ============================

DEF_CFG = {
    # Sentence-level split CSVs (disjoint by seed; no leakage)
    "train_csv": "data/splits/train_seed42.csv",
    "val_csv":   "data/splits/val_seed42.csv",
    "test_csv":  "data/splits/test_seed42.csv",

    # Sentence-level columns
    "TEXT_COL":     "covered_text",
    "LABEL_COL":    "GoldFaceAct",
    "EMAIL_COL":    "email_id",
    "SENTIDX_COL":  "sentence_idx",
    "ISREQ_COL":    "is_request",   # 1 = Request, 0 = Reply
    "SEED_COL":     "seed",

    # 9 FA labels (order fixed)
    "LABELS": ["HNeg+","HNeg-","HPos+","HPos-","Neutral","SNeg+","SNeg-","SPos+","SPos-"],

    # FA model (BERT + ASL) — TargetFirst PastSameEmail
    "fa_model_name": "bert-base-uncased",
    "max_length": 256,
    "fa_epochs": 5,
    "fa_batch_size": 16,
    "fa_lr": 2e-5,
    "fa_weight_decay": 0.01,
    "fa_early_stop": 2,
    "asl_gamma_pos": 0.0,
    "asl_gamma_neg": 4.0,
    "asl_clip": 0.05,
    "fa_use_pos_weight": False,
    "fa_force_tau_f1": 0.60,
    "fa_tmpdir": "./fa_models_past_same_email",

    # Doc table (targets)
    "DOC_CSV":  "data/corpus/email_text_gold_three_dimensions_politeness_score_with_seed_correct.csv",
    "DOC_ID_COL":   "email_id",
    "DOC_TEXT_COL": "text_email",
    "TARGETS": [
        "Directness_vs_Indirectness__GOLD",
        "Structural_Politeness_and_Politeness_Markers__GOLD",
        "Tone_and_Overall_Consideration__GOLD"
    ],

    # BERT doc-level (lean defaults)
    "bert_encoder": "bert-base-uncased",
    "bert_lr_grid": [2e-5],
    "bert_epochs_grid": [3, 5],
    "bert_dropout_grid": [0.1],
    "bert_batch_grid": [8],
    "chunk_len_grid": [400],
    "chunk_stride_grid": [350],

    # PredFA-only MLP grid
    "mlp_hidden_grid": [128],
    "mlp_dropout_grid": [0.1],
    "mlp_lr_grid": [1e-3],
    "mlp_epochs": 30,
    "mlp_batch": 256,

    # General
    "output_dir": "./end2end_outputs_hspt13_nogate_concat128",
    "seed": 42,

    # Early stopping for doc models
    "doc_early_stop_patience": 2,
    "doc_early_stop_min_delta": 1e-4,
    "doc_early_stop_lambda": 1e-3,   # small MAE regularizer weight

    # Runtime
    "num_workers": 4,
}

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def ensure_dir(d): os.makedirs(d, exist_ok=True)
def _safe_device(): return "cuda" if torch.cuda.is_available() else "cpu"
def _expand(p: str) -> str: return os.path.abspath(os.path.expanduser(p))
def _assert_file(p: str): assert os.path.isfile(p), f"Missing file: {p}"

def _ensure_writable_dir(p: str) -> str:
    p = _expand(p)
    try:
        os.makedirs(p, exist_ok=True)
        if os.access(p, os.W_OK):
            return p
    except Exception:
        pass
    fallback_root = _expand("~/.cache/opr")
    fallback = os.path.join(fallback_root, os.path.basename(p.rstrip(os.sep)) or "out")
    os.makedirs(fallback, exist_ok=True)
    if not os.access(fallback, os.W_OK):
        raise PermissionError(f"Cannot write to '{p}' and fallback '{fallback}' is not writable.")
    print(f"[WARN] '{p}' not writable. Using fallback '{fallback}'.")
    return fallback

def parse_labels(cell, LABELS):
    if cell is None or (isinstance(cell, float) and np.isnan(cell)): return []
    s = str(cell).strip()
    if s.startswith('[') and s.endswith(']'):
        try:
            arr = ast.literal_eval(s); return [str(x).strip() for x in arr]
        except Exception:
            pass
    return [t.strip() for t in re.split(r'[;,]', s) if t.strip()]

def to_multi_hot(names: List[str], label2id: Dict[str,int]) -> np.ndarray:
    v = np.zeros(len(label2id), dtype=np.float32)
    for n in names:
        if n in label2id: v[label2id[n]] = 1.0
    return v

def mae_rmse(y_true, y_pred):
    mae = np.mean(np.abs(y_true - y_pred), axis=0)
    rmse = np.sqrt(np.mean((y_true - y_pred)**2, axis=0))
    return mae, rmse

def spearman_each(y_true, y_pred):
    T = y_true.shape[1]
    rhos = []
    for t in range(T):
        rho, _ = spearmanr(y_true[:, t], y_pred[:, t])
        rhos.append(float(0.0 if np.isnan(rho) else rho))
    return rhos

def spearman_macro(y_true, y_pred):
    rhos = spearman_each(y_true, y_pred)
    return float(np.mean(rhos)), rhos

def _scrub(a: np.ndarray) -> np.ndarray:
    a = np.asarray(a, dtype=np.float32)
    a[~np.isfinite(a)] = 0.0
    return a

def sigmoid_stable_np(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    np.clip(x, -50.0, 50.0, out=x)
    return (1.0 / (1.0 + np.exp(-x))).astype(np.float32)

# ============================ Load sentence splits ============================

def load_sent_split(csv_path: str, CFG, filter_is_request: Union[int,None]):
    df = pd.read_csv(csv_path)
    need = [CFG["TEXT_COL"], CFG["LABEL_COL"], CFG["EMAIL_COL"], CFG["SENTIDX_COL"],
            CFG["ISREQ_COL"], CFG["SEED_COL"]]
    for c in need:
        assert c in df.columns, f"Missing '{c}' in {csv_path}"
    if filter_is_request in (0,1):
        df = df[df[CFG["ISREQ_COL"]] == filter_is_request].copy()

    for c in [CFG["EMAIL_COL"], CFG["SENTIDX_COL"], CFG["ISREQ_COL"], CFG["SEED_COL"]]:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)
    LABELS = CFG["LABELS"]; label2id = {l:i for i,l in enumerate(LABELS)}
    df["gold_list"] = df[CFG["LABEL_COL"]].map(lambda x: parse_labels(x, LABELS))
    df["y_vec"]     = df["gold_list"].map(lambda names: to_multi_hot(names, label2id))
    return df.reset_index(drop=True)

# ============================ PastSameEmail (TargetFirst) ============================

def add_target_and_context(df: pd.DataFrame, CFG) -> pd.DataFrame:
    email_col, sent_col, text_col = CFG["EMAIL_COL"], CFG["SENTIDX_COL"], CFG["TEXT_COL"]
    df2 = df.sort_values([email_col, sent_col]).copy()
    df2["target"]  = df2[text_col].astype(str)
    df2["context"] = ""
    for eid, grp in df2.groupby(email_col, sort=False):
        texts = grp[text_col].astype(str).tolist()
        ctx_vals, pasts = [], []
        for s in texts:
            ctx_vals.append(" ".join(pasts) if pasts else "")
            pasts.append(s)
        df2.loc[grp.index, "context"] = ctx_vals
    return df2.reset_index(drop=True)

class PastSameEmailDataset(Dataset):
    def __init__(self, df_ctx: pd.DataFrame, tok, max_length: int, label_vec_col="y_vec"):
        self.df = df_ctx.reset_index(drop=True)
        self.target  = self.df["target"].astype(str).tolist()
        self.context = self.df["context"].astype(str).tolist()
        self.labels  = np.stack(self.df[label_vec_col].values)  # [N, C]
        self.enc = tok(
            self.target, self.context,
            padding=True, truncation="only_second", max_length=max_length,
            return_token_type_ids=True, return_tensors=None
        )
    def __len__(self): return len(self.target)
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k,v in self.enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float32)
        return item

# ============================ ASL Loss & Trainer ============================

class AsymmetricLoss(nn.Module):
    def __init__(self, gamma_pos=0.0, gamma_neg=4.0, clip=0.05, eps=1e-8, reduction="mean"):
        super().__init__()
        self.gp, self.gn, self.clip, self.eps, self.reduction = gamma_pos, gamma_neg, clip, eps, reduction

    def forward(self, logits, targets, pos_weight=None):
        p = torch.sigmoid(logits)
        # Positive term
        log_pos = torch.log(p.clamp(min=self.eps))
        if self.gp > 0:
            log_pos = log_pos * (1.0 - p) ** self.gp
        # Negative term — clip on p, not on (1-p)
        pn = (p - self.clip).clamp_min(0.0) if (self.clip and self.clip > 0) else p
        log_neg = torch.log((1.0 - pn).clamp(min=self.eps))
        if self.gn > 0:
            log_neg = log_neg * (pn ** self.gn)

        loss = -(targets * log_pos + (1.0 - targets) * log_neg)

        if pos_weight is not None:
            loss = loss * (1.0 + targets * (pos_weight.to(loss.device) - 1.0))

        return loss.mean()

def training_metrics_fixed_tau(eval_pred):
    logits = getattr(eval_pred, "predictions", None)
    labels = getattr(eval_pred, "label_ids", None)
    if logits is None or labels is None:
        logits, labels = eval_pred
    probs = sigmoid_stable_np(logits)
    pred  = (probs >= 0.50).astype(int)
    return {
        "micro/f1":        f1_score(labels, pred, average="micro",  zero_division=0),
        "macro/f1":        f1_score(labels, pred, average="macro",  zero_division=0),
        "micro/precision": precision_score(labels, pred, average="micro", zero_division=0),
        "micro/recall":    recall_score(labels, pred, average="micro",  zero_division=0),
        "jaccard/micro":   jaccard_score(labels, pred, average="micro",  zero_division=0),
        "jaccard/macro":   jaccard_score(labels, pred, average="macro",  zero_division=0),
        "jaccard/samples": jaccard_score(labels, pred, average="samples", zero_division=0),
    }

class ASLTrainer(Trainer):
    def __init__(self, *args, pos_weight=None, asl_gamma_pos=0.0, asl_gamma_neg=4.0, asl_clip=0.05, **kwargs):
        super().__init__(*args, **kwargs)
        self.pos_weight = pos_weight
        self.asl = AsymmetricLoss(gamma_pos=asl_gamma_pos, gamma_neg=asl_gamma_neg, clip=asl_clip)
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels").float()
        outputs = model(**inputs)
        logits = outputs.logits
        loss = self.asl(logits, labels, pos_weight=self.pos_weight)
        return (loss, outputs) if return_outputs else loss

def _pos_weight_from_df(df_ctx: pd.DataFrame, LABELS: List[str], use_pos_weight: bool):
    if not use_pos_weight:
        return None
    y = np.stack(df_ctx["y_vec"].values)
    pos = y.sum(axis=0); neg = y.shape[0] - pos
    pos_weight = (neg / (pos + 1e-6)).astype(np.float32)
    return torch.tensor(pos_weight)

def pick_global_tau_by(metric_name, logits, labels, grid=np.linspace(0.05,0.95,19)):
    probs = sigmoid_stable_np(logits)
    best_tau, best_score = 0.5, -1
    for tau in grid:
        pred = (probs >= tau).astype(int)
        if metric_name == "micro/f1":
            score = f1_score(labels, pred, average="micro", zero_division=0)
        else:
            score = jaccard_score(labels, pred, average="micro", zero_division=0)
        if score > best_score:
            best_tau, best_score = float(tau), float(score)
    return best_tau, best_score

def train_fa_past_same_email(df_tr, df_va, df_te, CFG, out_dir="./fa_tmp", fast=False):
    LABELS = CFG["LABELS"]; label2id = {l:i for i,l in enumerate(LABELS)}

    def _prep(df):
        d = df.copy()
        d["gold_list"] = d[CFG["LABEL_COL"]].map(lambda x: parse_labels(x, LABELS))
        d["y_vec"]     = d["gold_list"].map(lambda names: to_multi_hot(names, label2id))
        return add_target_and_context(d, CFG)

    tr_ctx = _prep(df_tr); va_ctx = _prep(df_va); te_ctx = _prep(df_te)

    tok = AutoTokenizer.from_pretrained(CFG["fa_model_name"], use_fast=True)

    ds_tr = PastSameEmailDataset(tr_ctx, tok, max_length=CFG["max_length"])
    ds_va = PastSameEmailDataset(va_ctx, tok, max_length=CFG["max_length"])
    ds_te = PastSameEmailDataset(te_ctx, tok, max_length=CFG["max_length"])

    model = AutoModelForSequenceClassification.from_pretrained(
        CFG["fa_model_name"],
        num_labels=len(LABELS),
        problem_type="multi_label_classification",
        id2label={i:l for i,l in enumerate(LABELS)},
        label2id={l:i for i,l in enumerate(LABELS)}
    )

    has_cuda = torch.cuda.is_available()
    try:
        bf16_ok = has_cuda and torch.cuda.is_bf16_supported()
    except Exception:
        bf16_ok = False
    fp16_ok = has_cuda and not bf16_ok

    args = TrainingArguments(
        output_dir=out_dir,
        per_device_train_batch_size=CFG["fa_batch_size"],
        per_device_eval_batch_size=CFG["fa_batch_size"],
        learning_rate=CFG["fa_lr"],
        weight_decay=CFG["fa_weight_decay"],
        num_train_epochs=CFG["fa_epochs"],
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="micro/f1",
        greater_is_better=True,
        fp16=fp16_ok,
        bf16=bf16_ok,
        seed=CFG["seed"],
        report_to=[],
        logging_steps=100,
        remove_unused_columns=False,
        dataloader_num_workers=DEF_CFG["num_workers"],
        dataloader_pin_memory=True,
        save_total_limit=1
    )

    pos_weight_tensor = _pos_weight_from_df(tr_ctx, LABELS, CFG.get("fa_use_pos_weight", False))

    trainer = ASLTrainer(
        model=model, args=args, train_dataset=ds_tr, eval_dataset=ds_va, tokenizer=tok,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=CFG["fa_early_stop"])],
        compute_metrics=training_metrics_fixed_tau,
        pos_weight=pos_weight_tensor,
        asl_gamma_pos=CFG["asl_gamma_pos"], asl_gamma_neg=CFG["asl_gamma_neg"], asl_clip=CFG["asl_clip"]
    )

    print("TargetFirst: PastSameEmail — BERT pair + ASL")
    trainer.train()

    # ---- VAL threshold sweeps (force τ for print/eval)
    val_out = trainer.predict(ds_va)
    val_logits, val_labels = val_out.predictions, val_out.label_ids
    tau_f1_auto, _ = pick_global_tau_by("micro/f1", val_logits, val_labels)
    tau_f1 = float(CFG.get("fa_force_tau_f1", 0.60))
    print(f"[VAL][Global τ@F1] τ={tau_f1:.2f} (auto would be {tau_f1_auto:.2f})")

    return trainer, tok, ds_tr, ds_va, ds_te, tr_ctx, va_ctx, te_ctx, tau_f1, LABELS

def predict_sentence_probs_past_same_email(ds, trainer):
    out = trainer.predict(ds)
    logits = out.predictions
    probs = sigmoid_stable_np(logits)
    return probs, logits

def export_predfa_rows_from_ctx(df_ctx: pd.DataFrame, probs: np.ndarray, LABELS: List[str],
                                DOC_ID_COL: str, SENTIDX_COL: str):
    rows = []
    assert len(df_ctx) == probs.shape[0]
    for i in range(len(df_ctx)):
        row = {DOC_ID_COL: str(int(df_ctx.iloc[i][DOC_ID_COL])), SENTIDX_COL: int(df_ctx.iloc[i][SENTIDX_COL])}
        for k, lab in enumerate(LABELS):
            row[f"prob_{lab}"] = float(probs[i, k])
        rows.append(row)
    return rows

# ============================ PredFA aggregation — HSPT-13 ============================

def aggregate_predfa_hspt13(dfp: pd.DataFrame, DOC_ID_COL: str, SENTIDX_COL: str, LABELS: List[str]) -> pd.DataFrame:
    """
    HSPT-13 PredFA features per email:
      1–9:   sumprob_<label>   = Σ_k prob_<label>(sentence_k) for each FA label (incl. Neutral)
      10–11: sum_H, sum_S      = totals for all H* vs S* (exclude Neutral)
      12–13: sum_praise, sum_threat = totals for '+' vs '−' (exclude Neutral)
    """
    need = [DOC_ID_COL, SENTIDX_COL] + [f"prob_{l}" for l in LABELS]
    for c in need:
        assert c in dfp.columns, f"Missing PredFA column: {c}"

    dfp = dfp.copy()
    dfp[DOC_ID_COL] = dfp[DOC_ID_COL].astype(str)
    dfp[SENTIDX_COL] = pd.to_numeric(dfp[SENTIDX_COL], errors="coerce").fillna(0).astype(int)
    for l in LABELS:
        col = f"prob_{l}"
        dfp[col] = pd.to_numeric(dfp[col], errors="coerce").fillna(0.0).astype(float)

    dfp = dfp.sort_values([DOC_ID_COL, SENTIDX_COL], kind="mergesort", ignore_index=True)

    H_labels = ["HNeg+","HNeg-","HPos+","HPos-"]
    S_labels = ["SNeg+","SNeg-","SPos+","SPos-"]
    plus_labels  = ["HPos+","HNeg+","SPos+","SNeg+"]
    minus_labels = ["HPos-","HNeg-","SPos-","SNeg-"]

    rows = []
    for doc_id, g in dfp.groupby(DOC_ID_COL, sort=False):
        row = {DOC_ID_COL: str(doc_id)}
        # 1–9 per-label sums
        for l in LABELS:
            row[f"sumprob_{l}"] = float(g[f"prob_{l}"].sum())
        # 10–11 H/S
        row["sum_H"] = float(g[[f"prob_{l}" for l in H_labels]].sum().sum())
        row["sum_S"] = float(g[[f"prob_{l}" for l in S_labels]].sum().sum())
        # 12–13 praise/threat
        row["sum_praise"] = float(g[[f"prob_{l}" for l in plus_labels]].sum().sum())
        row["sum_threat"] = float(g[[f"prob_{l}" for l in minus_labels]].sum().sum())
        rows.append(row)
    return pd.DataFrame(rows)

# ============================ Doc-level datasets & model ============================

_DOC_TOKEN_CACHE: Dict[Tuple[str,str,int,int], Tuple[List[List[int]], List[List[int]]]] = {}

def _hash_text(text: str) -> str:
    return hashlib.md5(text.encode('utf-8')).hexdigest()

class DocDataset(Dataset):
    def __init__(self, df, text_col, targets, tokenizer, mode="simple",
                 max_len=512, chunk_len=400, chunk_stride=350,
                 predfa_df: Optional[pd.DataFrame]=None, id_col="email_id"):
        self.df = df.reset_index(drop=True); self.text_col=text_col; self.targets=targets; self.tk=tokenizer
        self.mode=mode; self.max_len=max_len; self.chunk_len=chunk_len; self.chunk_stride=chunk_stride
        self.id_col = id_col; self.predfa=None; self.predfa_cols=[]
        self.encoder_id = getattr(self.tk, "name_or_path", "tokenizer")
        if predfa_df is not None and id_col in predfa_df.columns:
            predfa_df = predfa_df.set_index(id_col)
            self.predfa = predfa_df.reindex(self.df[id_col].astype(str).values).reset_index(drop=True).fillna(0.0)
            self.predfa_cols = [c for c in self.predfa.columns if c != id_col]
    def __len__(self): return len(self.df)
    def _encode_chunks(self, text):
        key = (self.encoder_id, _hash_text(text), self.chunk_len, self.chunk_stride)
        if key in _DOC_TOKEN_CACHE:
            toks_list, att_list = _DOC_TOKEN_CACHE[key]
            return toks_list, att_list
        toks = self.tk.encode(text, add_special_tokens=False, truncation=False)
        if len(toks)==0: toks=[self.tk.unk_token_id]
        toks_list=[]; att_list=[]; i=0
        while i < len(toks):
            win = toks[i:i+self.chunk_len]
            win = [self.tk.cls_token_id] + win + [self.tk.sep_token_id]
            if len(win) > self.max_len:
                win = win[:self.max_len]
                if win[-1] != self.tk.sep_token_id: win[-1]=self.tk.sep_token_id
            attn = [1]*len(win)
            toks_list.append(win); att_list.append(attn)
            if i + self.chunk_len >= len(toks): break
            i += self.chunk_stride
        _DOC_TOKEN_CACHE[key] = (toks_list, att_list)
        return toks_list, att_list
    def __getitem__(self, idx):
        row = self.df.iloc[idx]; text = str(row[self.text_col]) if pd.notna(row[self.text_col]) else ""
        y = row[self.targets].values.astype(np.float32)
        if self.mode=="simple":
            enc = self.tk(text, max_length=self.max_len, truncation=True, padding='max_length', return_tensors='pt')
            ids = enc['input_ids'].squeeze(0).unsqueeze(0); att = enc['attention_mask'].squeeze(0).unsqueeze(0); K=1
        else:
            chunks, atts = self._encode_chunks(text)
            L=max(len(c) for c in chunks)
            ids=[]; att=[]
            for w,a in zip(chunks, atts):
                pad=L-len(w); ids.append(w+[self.tk.pad_token_id]*pad); att.append(a+[0]*pad)
            ids=torch.tensor(ids, dtype=torch.long); att=torch.tensor(att, dtype=torch.long); K=ids.size(0)
        item={"input_ids":ids, "attention_mask":att, "targets":torch.tensor(y, dtype=torch.float32), "n_chunks":K}
        if self.predfa is not None:
            feats = self.predfa.iloc[idx][self.predfa_cols].values.astype(np.float32)
            feats = _scrub(feats)
            item["predfa"]=torch.tensor(feats, dtype=torch.float32)
        else:
            item["predfa"]=None
        return item

def doc_collate(batch):
    max_K = max(b["input_ids"].size(0) for b in batch)
    L = max(b["input_ids"].size(1) for b in batch)
    def pad2(t, fill=0):
        if t.size(1) < L:
            pad_len = L - t.size(1)
            t = torch.cat([t, torch.full((t.size(0), pad_len), fill, dtype=t.dtype)], dim=1)
        if t.size(0) < max_K:
            pad_k = torch.full((max_K - t.size(0), L), fill, dtype=t.dtype)
            t = torch.cat([t, pad_k], dim=0)
        return t
    ids   = torch.stack([pad2(b["input_ids"], fill=0) for b in batch], dim=0)
    att   = torch.stack([pad2(b["attention_mask"], fill=0) for b in batch], dim=0)
    y     = torch.stack([b["targets"] for b in batch], dim=0)
    nk    = torch.tensor([b["n_chunks"] for b in batch], dtype=torch.long)

    predf_list = [b["predfa"] for b in batch]
    if all(p is None for p in predf_list):
        P = None
    else:
        maxD = max((p.numel() if p is not None else 0) for p in predf_list)
        rows=[]
        for p in predf_list:
            if p is None:
                rows.append(torch.zeros(maxD, dtype=torch.float32))
            elif p.numel() < maxD:
                rows.append(torch.cat([p, torch.zeros(maxD - p.numel())], dim=0))
            else:
                rows.append(p)
        P = torch.stack(rows, dim=0)
    return {"input_ids": ids, "attention_mask": att, "targets": y, "n_chunks": nk, "predfa": P}

# ============================ FUSION MODEL (concat → 128 ReLU → 1 per target) ============================

class BertDocRegressor(nn.Module):
    """
    Doc encoder (BERT) + optional PredFA fusion (no gating).
    Fusion:
        z = concat([doc_vec, predfa])         # predfa are HSPT-13 features
        h = Linear(z → 128) + ReLU + Dropout
        score_t = Linear(128 → 1)             # per target head
    If predfa is None or fusion_dim == 0:
        z = doc_vec
    """
    def __init__(self, encoder_name="bert-base-uncased", n_targets=3,
                 fusion_dim=0, dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(encoder_name)
        hid = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.fusion_dim = int(fusion_dim)
        in_dim = hid + (self.fusion_dim if self.fusion_dim > 0 else 0)

        self.hidden = nn.Sequential(
            nn.Linear(in_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        self.heads = nn.ModuleList([nn.Linear(128, 1) for _ in range(n_targets)])

    def forward(self, input_ids, attention_mask, n_chunks, predfa=None):
        B, K, L = input_ids.shape
        ids = input_ids.view(B*K, L); att = attention_mask.view(B*K, L)
        out = self.encoder(input_ids=ids, attention_mask=att, return_dict=True)
        pooled = out.pooler_output if (hasattr(out, "pooler_output") and out.pooler_output is not None) \
                 else out.last_hidden_state[:,0,:]
        H = pooled.size(-1); pooled = pooled.view(B, K, H)
        mask = (torch.arange(K, device=n_chunks.device).unsqueeze(0) < n_chunks.unsqueeze(1)).float().unsqueeze(-1)
        doc_vec = (pooled * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1.0)

        if (predfa is not None) and (self.fusion_dim > 0):
            if predfa.size(-1) != self.fusion_dim:
                D = predfa.size(-1)
                if D > self.fusion_dim:
                    predfa = predfa[..., :self.fusion_dim]
                else:
                    pad = torch.zeros(predfa.size(0), self.fusion_dim - D, device=predfa.device, dtype=predfa.dtype)
                    predfa = torch.cat([predfa, pad], dim=-1)
            z = torch.cat([doc_vec, predfa], dim=-1)
        else:
            z = doc_vec

        h = self.hidden(self.dropout(z))     # [B,128]
        outs = [head(h) for head in self.heads]  # list [B,1]
        return torch.cat(outs, dim=-1)           # [B, n_targets]

# ============================ Trainers (BERT / PredFA MLP) ============================

def train_eval_doc_bert(df_tr, df_va, df_te, CFG, text_col, targets, predfa_agg=None,
                        mode="simple", outdir="./doc_bert", use_predfa=False,
                        multi_task=True, num_workers=4):
    ensure_dir(outdir)
    tk = AutoTokenizer.from_pretrained(CFG["bert_encoder"], use_fast=True)
    device=_safe_device()
    lam = CFG.get("doc_early_stop_lambda", 1e-3)
    ID_COL = CFG["DOC_ID_COL"]

    def run_once(trg_list, outdir_run):
        ensure_dir(outdir_run)
        global_best_score = -1e9
        global_best_state = None
        global_best_cfg   = None

        for lr in CFG["bert_lr_grid"]:
            for ep in CFG["bert_epochs_grid"]:
                for dr in CFG["bert_dropout_grid"]:
                    for bs in CFG["bert_batch_grid"]:
                        for chL in ([None] if mode=="simple" else CFG["chunk_len_grid"]):
                            for chS in ([None] if mode=="simple" else CFG["chunk_stride_grid"]):
                                print(f"[BERT] try lr={lr} ep={ep} drop={dr} bs={bs} mode={mode} predfa={use_predfa}")
                                ds_tr = DocDataset(df_tr, text_col, trg_list, tk, mode=mode,
                                                   max_len=512, chunk_len=chL or 400, chunk_stride=chS or 350,
                                                   predfa_df=predfa_agg if use_predfa else None,
                                                   id_col=ID_COL)
                                ds_va = DocDataset(df_va, text_col, trg_list, tk, mode=mode,
                                                   max_len=512, chunk_len=chL or 400, chunk_stride=chS or 350,
                                                   predfa_df=predfa_agg if use_predfa else None,
                                                   id_col=ID_COL)
                                dl_tr = DataLoader(ds_tr, batch_size=bs, shuffle=True,  collate_fn=doc_collate,
                                                   pin_memory=True, num_workers=num_workers)
                                dl_va = DataLoader(ds_va, batch_size=bs, shuffle=False, collate_fn=doc_collate,
                                                   pin_memory=True, num_workers=num_workers)

                                fusion_dim = 0
                                if use_predfa and predfa_agg is not None:
                                    sample = ds_tr[0]
                                    fusion_dim = sample["predfa"].numel()

                                model = BertDocRegressor(encoder_name=CFG["bert_encoder"], n_targets=len(trg_list),
                                                         fusion_dim=fusion_dim, dropout=dr).to(device)
                                opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
                                total_steps = max(1, len(dl_tr)*ep)
                                warmup_steps = int(0.06*total_steps)
                                sch = get_linear_schedule_with_warmup(opt, warmup_steps, total_steps)
                                loss_fn = nn.SmoothL1Loss()

                                patience = CFG.get("doc_early_stop_patience", 2)
                                min_delta = CFG.get("doc_early_stop_min_delta", 1e-4)
                                best_val = -1e9
                                best_state = None
                                bad_epochs = 0

                                for epoch in range(1, ep+1):
                                    model.train()
                                    for batch in dl_tr:
                                        opt.zero_grad()
                                        ids=batch["input_ids"].to(device); att=batch["attention_mask"].to(device)
                                        tg=batch["targets"].to(device); nk=batch["n_chunks"].to(device)
                                        pf=batch["predfa"].to(device) if batch["predfa"] is not None else None
                                        preds = model(ids, att, nk, pf); loss=loss_fn(preds, tg)
                                        loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
                                        opt.step(); sch.step()

                                    model.eval(); Ys=[]; Ps=[]
                                    with torch.no_grad():
                                        for batch in dl_va:
                                            ids=batch["input_ids"].to(device); att=batch["attention_mask"].to(device)
                                            tg=batch["targets"].to(device); nk=batch["n_chunks"].to(device)
                                            pf=batch["predfa"].to(device) if batch["predfa"] is not None else None
                                            preds = model(ids, att, nk, pf)
                                            Ys.append(tg.cpu().numpy()); Ps.append(preds.cpu().numpy())
                                    Yv=np.concatenate(Ys,0); Pv=np.concatenate(Ps,0)
                                    rho_macro, rhos = spearman_macro(Yv, Pv)
                                    mae_val, _ = mae_rmse(Yv, Pv)
                                    score = rho_macro - lam * float(np.mean(mae_val))

                                    if epoch == 1 and best_val == -1e9 and score < -0.5:
                                        print("[BERT] Prune candidate (weak after epoch 1).")
                                        break

                                    if score > best_val + min_delta:
                                        best_val = score
                                        bad_epochs = 0
                                        best_state = {
                                            "model": model.state_dict(),
                                            "cfg": {"lr":lr,"epochs":epoch,"dropout":dr,"batch":bs,"mode":mode,
                                                    "chunk_len":chL,"chunk_stride":chS,"use_predfa":use_predfa,
                                                    "targets":trg_list}
                                        }
                                    else:
                                        bad_epochs += 1
                                        if bad_epochs >= patience:
                                            print(f"[BERT] Early stop at epoch {epoch} (best score={best_val:.4f}).")
                                            break

                                if best_state is not None and best_val > global_best_score:
                                    global_best_score = best_val
                                    global_best_state = best_state
                                    global_best_cfg   = best_state["cfg"]

        if global_best_state is None:
            raise RuntimeError("BERT doc tuning produced no runs (check dataset sizes).")

        ds_te = DocDataset(
            df_te, text_col, trg_list, tk, mode=global_best_cfg["mode"],
            max_len=512, chunk_len=global_best_cfg["chunk_len"] or 400,
            chunk_stride=global_best_cfg["chunk_stride"] or 350,
            predfa_df=predfa_agg if global_best_cfg["use_predfa"] else None, id_col=ID_COL
        )
        fusion_dim = 0
        if global_best_cfg["use_predfa"] and predfa_agg is not None:
            sample = ds_te[0]; fusion_dim = sample["predfa"].numel()

        model = BertDocRegressor(encoder_name=CFG["bert_encoder"], n_targets=len(trg_list),
                                 fusion_dim=fusion_dim, dropout=global_best_cfg["dropout"]).to(device)
        model.load_state_dict(global_best_state["model"])
        model.eval()

        dl_te = DataLoader(ds_te, batch_size=global_best_cfg["batch"], shuffle=False,
                           collate_fn=doc_collate, pin_memory=True, num_workers=num_workers)
        Ys=[]; Ps=[]
        with torch.no_grad():
            for batch in dl_te:
                ids=batch["input_ids"].to(device); att=batch["attention_mask"].to(device)
                tg=batch["targets"].to(device); nk=batch["n_chunks"].to(device)
                pf=batch["predfa"].to(device) if batch["predfa"] is not None else None
                preds = model(ids, att, nk, pf); Ys.append(tg.cpu().numpy()); Ps.append(preds.cpu().numpy())
        Yt=np.concatenate(Ys,0); Pt=np.concatenate(Ps,0)
        mae_te, rmse_te = mae_rmse(Yt, Pt)
        rhos_te = spearman_each(Yt, Pt)
        rho_macro = float(np.mean(rhos_te))

        ensure_dir(outdir_run)
        with open(os.path.join(outdir_run, "metrics.json"), "w") as f:
            json.dump({
                "best_cfg": global_best_cfg,
                "dev_best_score": global_best_score,
                "test": {"MAE": mae_te.tolist(), "RMSE": rmse_te.tolist(),
                         "rho_macro": rho_macro, "rhos": rhos_te},
                "targets": trg_list
            }, f, indent=2)
        np.save(os.path.join(outdir_run,"test_y.npy"), Yt)
        np.save(os.path.join(outdir_run,"test_pred.npy"), Pt)
        return Yt, Pt, mae_te.tolist(), rho_macro, rhos_te

    # -------- multi-task OR single-task wrapper --------
    if multi_task:
        Y, P, MAE, rho_macro, rhos = run_once(targets, outdir+"_mt")
        return {"Y":Y, "P":P, "MAE":MAE, "rho_macro":rho_macro, "rhos":rhos, "variant":"mt"}

    # single-task: run per target (each model has 1 head), then stitch
    all_P=[]; all_Y=None; maes=[]; rhos=[]
    for ti, tgt in enumerate(targets):
        Yt, Pt, MAE, rho, rhos_single = run_once([tgt], outdir+"_st_"+str(ti))
        if all_Y is None: all_Y = Yt
        all_P.append(Pt); maes.append(MAE[0]); rhos.append(rhos_single[0])
    P = np.concatenate(all_P, axis=1); rho_macro = float(np.mean(rhos))
    ensure_dir(outdir+"_st")
    with open(os.path.join(outdir+"_st", "metrics.json"), "w") as f:
        json.dump({
            "best_cfg": "per-target single-task (see *_st_0,1,2)",
            "test": {"MAE": maes, "rho_macro": rho_macro, "rhos": rhos},
            "targets": targets
        }, f, indent=2)
    np.save(os.path.join(outdir+"_st","test_y.npy"), all_Y)
    np.save(os.path.join(outdir+"_st","test_pred.npy"), P)
    return {"Y":all_Y, "P":P, "MAE":maes, "rho_macro":rho_macro, "rhos":rhos, "variant":"st"}

# -------------------- PredFA-only MLP (uses HSPT-13) --------------------

class PredFAMLP(nn.Module):
    def __init__(self, in_dim, out_dim, hidden=128, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, out_dim)
        )
    def forward(self, x): return self.net(x)

def run_predfa_mlp(df_tr_doc, df_va_doc, df_te_doc, F_tr, F_va, F_te, targets, CFG, outdir, multi_task=True):
    def align(df_doc, F):
        F = F.copy()
        assert CFG["DOC_ID_COL"] in F.columns
        return F.set_index(CFG["DOC_ID_COL"]).reindex(df_doc[CFG["DOC_ID_COL"]].astype(str).values).fillna(0.0).reset_index(drop=True)

    ensure_dir(outdir)
    feat_cols = [c for c in F_tr.columns if c != CFG["DOC_ID_COL"]]

    def _train_epoch_loop(model, Xtr, Ytr, Xva, Yva, device, lr, epochs, patience, min_delta):
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        loss_fn = nn.SmoothL1Loss()
        idx = torch.arange(Xtr.size(0))

        best_score = -1e9
        best_state = None
        bad_epochs = 0
        lam = CFG.get("doc_early_stop_lambda", 1e-3)

        for ep in range(1, epochs+1):
            model.train()
            perm = idx[torch.randperm(idx.numel())]
            for i in range(0, perm.numel(), CFG["mlp_batch"]):
                b = perm[i:i+CFG["mlp_batch"]]
                xb = Xtr[b].to(device); yb=Ytr[b].to(device)
                opt.zero_grad(); pred = model(xb); loss = loss_fn(pred, yb)
                loss.backward(); opt.step()

            model.eval()
            with torch.no_grad():
                Pv = model(Xva.to(device)).cpu().numpy()
            rho_macro, _ = spearman_macro(Yva.numpy(), Pv)
            mae_val, _ = mae_rmse(Yva.numpy(), Pv)
            score = rho_macro - lam * float(np.mean(mae_val))

            if (best_state is None) or (score > best_score + min_delta):
                best_score = score
                best_state = model.state_dict()
                bad_epochs = 0
            else:
                bad_epochs += 1
                if bad_epochs >= patience:
                    print(f"[MLP] Early stop at epoch {ep} (best score={best_score:.4f}).")
                    break

        return best_state if best_state is not None else model.state_dict(), best_score

    def _train_or_retry_then_cpu(model, Xtr, Ytr, Xva, Yva, device, lr, epochs, patience, min_delta):
        tried_cuda_retry = False
        try:
            return _train_epoch_loop(model.to(device), Xtr, Ytr, Xva, Yva, device, lr, epochs, patience, min_delta)
        except RuntimeError as e:
            if ("CUDA" in str(e)) and (device=="cuda") and (not tried_cuda_retry):
                print("[WARN] CUDA hiccup in PredFA MLP — retrying once on CUDA.")
                tried_cuda_retry = True
                try: torch.cuda.empty_cache()
                except Exception: pass
                return _train_epoch_loop(model.to("cuda"), Xtr, Ytr, Xva, Yva, "cuda", lr, epochs, patience, min_delta)
            print("[WARN] Falling back to CPU for PredFA MLP.")
            return _train_epoch_loop(model.to("cpu"), Xtr, Ytr, Xva, Yva, "cpu", lr, epochs, patience, min_delta)

    def run_once(trg_list, outdir_run):
        ensure_dir(outdir_run)
        Ft = _scrub(align(df_tr_doc, F_tr)[feat_cols].to_numpy(np.float32))
        Fv = _scrub(align(df_va_doc, F_va)[feat_cols].to_numpy(np.float32))
        Fe = _scrub(align(df_te_doc, F_te)[feat_cols].to_numpy(np.float32))
        Yt = _scrub(df_tr_doc[trg_list].to_numpy(np.float32))
        Yv = _scrub(df_va_doc[trg_list].to_numpy(np.float32))
        Ye = _scrub(df_te_doc[trg_list].to_numpy(np.float32))

        Xtr = torch.tensor(Ft); Xva = torch.tensor(Fv); Xte = torch.tensor(Fe)
        Ytr = torch.tensor(Yt); Yva = torch.tensor(Yv); Yte = torch.tensor(Ye)
        best=None; device=_safe_device()

        patience = CFG.get("doc_early_stop_patience", 2)
        min_delta = CFG.get("doc_early_stop_min_delta", 1e-4)

        for hidden in CFG["mlp_hidden_grid"]:
            for dr in CFG["mlp_dropout_grid"]:
                for lr in CFG["mlp_lr_grid"]:
                    model = PredFAMLP(Xtr.size(1), len(trg_list), hidden=hidden, dropout=dr)
                    state, val_score = _train_or_retry_then_cpu(
                        model, Xtr, Ytr, Xva, Yva, device, lr,
                        epochs=CFG["mlp_epochs"], patience=patience, min_delta=min_delta
                    )
                    cand=(val_score, hidden, dr, lr, state)
                    if (best is None) or (val_score>best[0]): best=cand

        _, hidden, dr, lr, state = best
        final_device=_safe_device()
        model = PredFAMLP(Xtr.size(1), len(trg_list), hidden=hidden, dropout=dr).to(final_device)
        model.load_state_dict(state); model.eval()
        with torch.no_grad():
            Pt = model(Xte.to(final_device)).cpu().numpy()
        mae_te, rmse_te = mae_rmse(Ye, Pt)
        rhos_te = spearman_each(Ye, Pt)
        rho_macro = float(np.mean(rhos_te))

        with open(os.path.join(outdir_run, "metrics.json"), "w") as f:
            json.dump({
                "best_cfg": {"hidden": hidden, "dropout": dr, "lr": lr},
                "test": {"MAE": mae_te.tolist(), "RMSE": rmse_te.tolist(),
                         "rho_macro": rho_macro, "rhos": rhos_te},
                "targets": trg_list
            }, f, indent=2)
        np.save(os.path.join(outdir_run,"test_y.npy"), Ye)
        np.save(os.path.join(outdir_run,"test_pred.npy"), Pt)
        return Ye, Pt, mae_te.tolist(), rho_macro, rhos_te

    if multi_task:
        Y, P, MAE, rho_macro, rhos = run_once(targets, outdir+"_mt")
        return {"Y":Y, "P":P, "MAE":MAE, "rho_macro":rho_macro, "rhos":rhos, "variant":"mt"}

    all_P=[]; all_Y=None; maes=[]; rhos=[]
    for ti, tgt in enumerate(targets):
        Yt, Pt, MAE, rho, rhos_single = run_once([tgt], outdir+"_st_"+str(ti))
        if all_Y is None: all_Y = Yt
        all_P.append(Pt); maes.append(MAE[0]); rhos.append(rhos_single[0])
    P = np.concatenate(all_P, axis=1); rho_macro = float(np.mean(rhos))
    ensure_dir(outdir+"_st")
    with open(os.path.join(outdir+"_st", "metrics.json"), "w") as f:
        json.dump({
            "best_cfg": "per-target single-task (see *_st_0,1,2)",
            "test": {"MAE": maes, "rho_macro": rho_macro, "rhos": rhos},
            "targets": targets
        }, f, indent=2)
    np.save(os.path.join(outdir+"_st","test_y.npy"), all_Y)
    np.save(os.path.join(outdir+"_st","test_pred.npy"), P)
    return {"Y":all_Y, "P":P, "MAE":maes, "rho_macro":rho_macro, "rhos":rhos, "variant":"st"}

# ============================ Main ============================

def main():
    ap = argparse.ArgumentParser()
    # paths
    ap.add_argument("--train_csv", type=str, required=True)
    ap.add_argument("--val_csv",   type=str, required=True)
    ap.add_argument("--test_csv",  type=str, required=True)
    ap.add_argument("--doc_csv",   type=str, required=True)
    # FA filter (Request=1 / Reply=0 / both if omitted)
    ap.add_argument("--fa_filter_is_request", type=int, choices=[0,1], default=None,
                    help="1=requests only, 0=replies only, omit=both")
    # toggles (families). We will ALWAYS run both MT and ST variants in one go.
    ap.add_argument("--do_doc_bert", action="store_true")
    ap.add_argument("--do_predfa_mlp", action="store_true")
    ap.add_argument("--use_predfa_fusion", action="store_true")
    ap.add_argument("--bert_mode", choices=["simple","hier"], default="simple")
    ap.add_argument("--skip_fa", action="store_true")
    ap.add_argument("--fast", action="store_true", help="Faster run settings")
    # general
    ap.add_argument("--outdir", type=str, default=DEF_CFG["output_dir"])
    args, _unknown = ap.parse_known_args()

    # ---------- PATH VALIDATION & WRITABLE FALLBACKS ----------
    args.train_csv = _expand(args.train_csv)
    args.val_csv   = _expand(args.val_csv)
    args.test_csv  = _expand(args.test_csv)
    args.doc_csv   = _expand(args.doc_csv)
    for p in [args.train_csv, args.val_csv, args.test_csv, args.doc_csv]:
        _assert_file(p)

    args.outdir = _ensure_writable_dir(args.outdir)
    DEF_CFG["fa_tmpdir"] = _ensure_writable_dir(os.path.join(args.outdir, "fa_models_past_same_email"))

    set_seed(DEF_CFG["seed"])
    CFG = DEF_CFG.copy()
    CFG["train_csv"]=args.train_csv; CFG["val_csv"]=args.val_csv; CFG["test_csv"]=args.test_csv
    CFG["DOC_CSV"]=args.doc_csv;    CFG["output_dir"]=args.outdir

    # FAST mode tweaks
    if args.fast:
        print("[FAST MODE] Enabled — condensed grids and shorter seqs.")
        CFG["fa_model_name"] = "bert-base-uncased"
        CFG["bert_encoder"] = "bert-base-uncased"
        CFG["max_length"] = 192
        CFG["fa_epochs"] = 3
        CFG["fa_batch_size"] = 16
        CFG["bert_lr_grid"] = [2e-5]
        CFG["bert_epochs_grid"] = [3]
        CFG["bert_dropout_grid"] = [0.1]
        CFG["bert_batch_grid"] = [8]
        CFG["chunk_len_grid"] = [320]
        CFG["chunk_stride_grid"] = [256]
        CFG["mlp_hidden_grid"] = [128]
        CFG["mlp_dropout_grid"] = [0.1]
        CFG["mlp_lr_grid"] = [1e-3]
        CFG["mlp_epochs"] = 20
        CFG["mlp_batch"] = 512

    # ===== Sentence splits
    df_train = load_sent_split(CFG["train_csv"], CFG, args.fa_filter_is_request)
    df_val   = load_sent_split(CFG["val_csv"],   CFG, args.fa_filter_is_request)
    df_test  = load_sent_split(CFG["test_csv"],  CFG, args.fa_filter_is_request)

    # Guard: no seed overlap
    s_tr, s_va, s_te = set(df_train[CFG["SEED_COL"]]), set(df_val[CFG["SEED_COL"]]), set(df_test[CFG["SEED_COL"]])
    assert not (s_tr & s_va) and not (s_tr & s_te) and not (s_va & s_te), "Leakage: seed overlap across splits"

    LABELS = CFG["LABELS"]

    # ===== Phase A: FA → PredFA CSVs (TargetFirst PastSameEmail)
    train_pred_csv = os.path.join(CFG["output_dir"], "predfa_train.csv")
    val_pred_csv   = os.path.join(CFG["output_dir"], "predfa_val.csv")
    test_pred_csv  = os.path.join(CFG["output_dir"], "predfa_test.csv")

    if not args.skip_fa:
        print("[FA][PastSameEmail] Building datasets (TRAIN/VAL/TEST)…")
        trainer, tok, ds_tr, ds_va, ds_te, tr_ctx, va_ctx, te_ctx, tau_f1, LABELS_used = train_fa_past_same_email(
            df_train, df_val, df_test, CFG,
            out_dir=os.path.join(CFG["fa_tmpdir"], "trainval"),
            fast=args.fast
        )
        print(f"[FA] Forced Global τ@F1 used for reporting/eval: τ={tau_f1:.2f}")

        # TRAIN preds
        print("[FA] Predicting TRAIN…")
        tr_probs, _ = predict_sentence_probs_past_same_email(ds_tr, trainer)
        tr_rows = export_predfa_rows_from_ctx(tr_ctx, tr_probs, LABELS_used, CFG["DOC_ID_COL"], CFG["SENTIDX_COL"])
        pd.DataFrame(tr_rows).to_csv(train_pred_csv, index=False); print("[FA] Saved:", train_pred_csv)

        # VAL preds
        print("[FA] Predicting VAL…")
        va_probs, _ = predict_sentence_probs_past_same_email(ds_va, trainer)
        va_rows = export_predfa_rows_from_ctx(va_ctx, va_probs, LABELS_used, CFG["DOC_ID_COL"], CFG["SENTIDX_COL"])
        pd.DataFrame(va_rows).to_csv(val_pred_csv, index=False); print("[FA] Saved:", val_pred_csv)

        # TEST preds
        print("[FA] Predicting TEST…")
        te_probs, _ = predict_sentence_probs_past_same_email(ds_te, trainer)
        te_rows = export_predfa_rows_from_ctx(te_ctx, te_probs, LABELS_used, CFG["DOC_ID_COL"], CFG["SENTIDX_COL"])
        pd.DataFrame(te_rows).to_csv(test_pred_csv, index=False); print("[FA] Saved:", test_pred_csv)
    else:
        assert all(os.path.isfile(p) for p in [train_pred_csv, val_pred_csv, test_pred_csv]), \
            "skip_fa set but PredFA CSVs not found."

    # ===== Phase B: HSPT-13 aggregation
    print("[PredFA] Aggregating HSPT-13 (9 per-label + H/S + praise/threat)…")
    F_tr = aggregate_predfa_hspt13(pd.read_csv(train_pred_csv), CFG["DOC_ID_COL"], CFG["SENTIDX_COL"], LABELS)
    F_va = aggregate_predfa_hspt13(pd.read_csv(val_pred_csv),   CFG["DOC_ID_COL"], CFG["SENTIDX_COL"], LABELS)
    F_te = aggregate_predfa_hspt13(pd.read_csv(test_pred_csv),  CFG["DOC_ID_COL"], CFG["SENTIDX_COL"], LABELS)

    # Z-score the 13 features using TRAIN μ/σ (keep id column intact)
    feat_cols = [c for c in F_tr.columns if c != CFG["DOC_ID_COL"]]
    assert len(feat_cols) == 13, f"Expected 13 features, got {len(feat_cols)}"

    mu = F_tr[feat_cols].mean(axis=0)
    sd = F_tr[feat_cols].std(axis=0).replace(0, 1.0)  # guard against zero std

    F_tr_z = F_tr.copy(); F_tr_z[feat_cols] = (F_tr[feat_cols] - mu) / sd
    F_va_z = F_va.copy(); F_va_z[feat_cols] = (F_va[feat_cols] - mu) / sd
    F_te_z = F_te.copy(); F_te_z[feat_cols] = (F_te[feat_cols] - mu) / sd


    # ===== Phase C: Doc CSV and splits
    df_doc_all = pd.read_csv(CFG["DOC_CSV"]).copy()
    for c in [CFG["DOC_ID_COL"], CFG["DOC_TEXT_COL"]] + CFG["TARGETS"]:
        assert c in df_doc_all.columns, f"Missing '{c}' in DOC_CSV"
    df_doc_all[CFG["DOC_ID_COL"]] = df_doc_all[CFG["DOC_ID_COL"]].astype(str)

    ids_tr = set(df_train[CFG["EMAIL_COL"]].astype(int).astype(str).unique())
    ids_va = set(df_val  [CFG["EMAIL_COL"]].astype(int).astype(str).unique())
    ids_te = set(df_test [CFG["EMAIL_COL"]].astype(int).astype(str).unique())

    df_tr_doc = df_doc_all[df_doc_all[CFG["DOC_ID_COL"]].isin(ids_tr)].reset_index(drop=True)
    df_va_doc = df_doc_all[df_doc_all[CFG["DOC_ID_COL"]].isin(ids_va)].reset_index(drop=True)
    df_te_doc = df_doc_all[df_doc_all[CFG["DOC_ID_COL"]].isin(ids_te)].reset_index(drop=True)

    # Fusion dataframe (HSPT-13; keep email_id column)
    for F in (F_tr, F_va, F_te):
        F[CFG["DOC_ID_COL"]] = F[CFG["DOC_ID_COL"]].astype(str)
    fusion_df_all = pd.concat([F_tr_z.copy(), F_va_z.copy(), F_te_z.copy()], axis=0, ignore_index=True)

    # Sanity checks
    def _check_align(df_doc, F_pred, idcol):
        a = set(df_doc[idcol]); b = set(F_pred[idcol])
        missing = a - b
        assert len(missing) == 0, f"PredFA (HSPT-13) missing {len(missing)} ids; e.g., {list(missing)[:5]}"
    _check_align(df_tr_doc, F_tr, CFG["DOC_ID_COL"])
    _check_align(df_va_doc, F_va, CFG["DOC_ID_COL"])
    _check_align(df_te_doc, F_te, CFG["DOC_ID_COL"])

    # We ALWAYS produce both MT and ST variants in ONE run.
    results = {}

    # ---- 1) BERT (text-only)
    if args.do_doc_bert:
        print("\n[DOC] BERT (text-only)…")
        results["BERT_text_mt"] = train_eval_doc_bert(
            df_tr_doc, df_va_doc, df_te_doc, CFG, CFG["DOC_TEXT_COL"], CFG["TARGETS"],
            predfa_agg=None, mode=args.bert_mode,
            outdir=os.path.join(CFG["output_dir"], f"bert_textonly_{args.bert_mode}"),
            use_predfa=False, multi_task=True, num_workers=CFG["num_workers"]
        )
        results["BERT_text_st"] = train_eval_doc_bert(
            df_tr_doc, df_va_doc, df_te_doc, CFG, CFG["DOC_TEXT_COL"], CFG["TARGETS"],
            predfa_agg=None, mode=args.bert_mode,
            outdir=os.path.join(CFG["output_dir"], f"bert_textonly_{args.bert_mode}"),
            use_predfa=False, multi_task=False, num_workers=CFG["num_workers"]
        )

    if torch.cuda.is_available():
        try: torch.cuda.empty_cache()
        except Exception: pass

    # ---- 2) PredFA-only (MLP) — uses HSPT-13  F_tr_z.copy(), F_va_z.copy(), F_te_z.copy()
    if args.do_predfa_mlp:
        print("\n[DOC] PredFA-only (MLP, HSPT-13)…")
        results["PredFA_only_mt"] = run_predfa_mlp(
            df_tr_doc, df_va_doc, df_te_doc, F_tr_z.copy(), F_va_z.copy(), F_te_z.copy(),
            CFG["TARGETS"], CFG, outdir=os.path.join(CFG["output_dir"], "predfa_mlp_hspt13"), multi_task=True
        )
        results["PredFA_only_st"] = run_predfa_mlp(
            df_tr_doc, df_va_doc, df_te_doc, F_tr_z.copy(), F_va_z.copy(), F_te_z.copy(),
            CFG["TARGETS"], CFG, outdir=os.path.join(CFG["output_dir"], "predfa_mlp_hspt13"), multi_task=False
        )

    # ---- 3) BERT + PredFA (simple concat fusion; HSPT-13)
    if args.use_predfa_fusion:
        print("\n[DOC] BERT + PredFA (concat fusion, HSPT-13)…")
        fusion_df = fusion_df_all
        results["BERT_plus_PredFA_mt"] = train_eval_doc_bert(
            df_tr_doc, df_va_doc, df_te_doc, CFG, CFG["DOC_TEXT_COL"], CFG["TARGETS"],
            predfa_agg=fusion_df, mode=args.bert_mode,
            outdir=os.path.join(CFG["output_dir"], f"bert_text_predfa_hspt13_{args.bert_mode}"),
            use_predfa=True, multi_task=True, num_workers=CFG["num_workers"]
        )
        results["BERT_plus_PredFA_st"] = train_eval_doc_bert(
            df_tr_doc, df_va_doc, df_te_doc, CFG, CFG["DOC_TEXT_COL"], CFG["TARGETS"],
            predfa_agg=fusion_df, mode=args.bert_mode,
            outdir=os.path.join(CFG["output_dir"], f"bert_text_predfa_hspt13_{args.bert_mode}"),
            use_predfa=True, multi_task=False, num_workers=CFG["num_workers"]
        )

    # ---- Save comparison table + pretty print with per-target Spearman + MAE
    def rows_for(tag, r):
        return {
            f"{tag}_Directness_MAE": r["MAE"][0],
            f"{tag}_Markers_MAE":   r["MAE"][1],
            f"{tag}_Overall_MAE":   r["MAE"][2],
            f"{tag}_rho_D":         r["rhos"][0],
            f"{tag}_rho_M":         r["rhos"][1],
            f"{tag}_rho_O":         r["rhos"][2],
            f"{tag}_rho_macro":     r["rho_macro"]
        }

    compare = {}
    for k,v in results.items():
        compare.update(rows_for(k, v))

    with open(os.path.join(CFG["output_dir"], "compare_table.json"), "w") as f:
        json.dump(compare, f, indent=2)

    def pr_line(tag, r):
        mae = r["MAE"]; rhos = r["rhos"]
        print(f"{tag:28s} | MAE ↓  D:{mae[0]:.4f} M:{mae[1]:.4f} O:{mae[2]:.4f} | ρ_D:{rhos[0]:.4f} ρ_M:{rhos[1]:.4f} ρ_O:{rhos[2]:.4f} (macro {r['rho_macro']:.4f})")

    if results:
        print("\n=== TEST Comparison (lower MAE better, higher ρ better) ===")
        for k in sorted(results.keys()):
            pr_line(k, results[k])
    print("\n[DONE] One run produced both multi-task and single-task outputs.")

if __name__ == "__main__":
    main()
